In [23]:
import re
import json
from collections import defaultdict
from typing import List, Dict, Optional

# 1. DEFINICIÓN DE PATRONES (Uso de re.VERBOSE para claridad total)
# Usamos r'' para cadenas crudas y los comentarios ayudan a saber qué hace cada línea
PATRON_HTTP = re.compile(r'''
    ^                                       # Inicio
    (?P<ip>\d{1,3}(?:\.\d{1,3}){3})         # Captura IP
    \s+-\s+-\s+                             # Espaciado
    \[(?P<timestamp>[^\]]+)\]               # Tiempo
    \s+"(?P<method>[A-Z]+)\s+(?P<path>[^\s]+)\s+HTTP/[0-9.]+" # Método y ruta
    \s+(?P<status>\d{3})                    # Estado
    \s+(?P<bytes>\d+|-)                     # Tamaño
''', re.VERBOSE)

PATRON_AUTH = re.compile(r'''
    ^\[AUTH\]\s+(?P<timestamp>[\d-]+\s[\d:]+)\s+\|\s+
    (?<=user=)(?P<user>[^\s|]+)\s+\|\s+
    (?<=action=)(?P<action>[^\s|]+)\s+\|\s+
    (?<=status=)(?P<status>[^\s|]+)
''', re.VERBOSE)

# 2. FUNCIONES DE PARSEO (Limpias y reutilizables)
def parse_http(linea: str) -> Optional[Dict]:
    m = PATRON_HTTP.search(linea)
    if m:
        d = m.groupdict()
        d['status'] = int(d['status'])
        d['bytes'] = int(d['bytes']) if d['bytes'] != '-' else 0
        return d
    return None

def parse_auth(linea: str) -> Optional[Dict]:
    m = PATRON_AUTH.search(linea)
    return m.groupdict() if m else None

# 3. ANALIZADOR DE SEGURIDAD (Modular)
def analizar_seguridad(auth_logs: List[Dict]) -> Dict:
    # Detectar fuerza bruta (más de 3 intentos fallidos)
    intentos = defaultdict(int)
    for log in auth_logs:
        if log['status'] == 'FAILED':
            intentos[log['ip']] += 1
    
    return {
        "sospechosos_fuerza_bruta": [ip for ip, count in intentos.items() if count > 3]
    }

# 4. PROCESADOR PRINCIPAL (El motor)
def procesar_logs(nombre_archivo: str):
    reporte = {"http": [], "auth": [], "seguridad": {}}
    
    with open(nombre_archivo, 'r') as f:
        for linea in f:
            linea = linea.strip()
            # Clasificar y parsear
            if "HTTP" in linea:
                res = parse_http(linea)
                if res: reporte["http"].append(res)
            elif "[AUTH]" in linea:
                res = parse_auth(linea)
                if res: reporte["auth"].append(res)
    
    # Análisis final
    reporte["seguridad"] = analizar_seguridad(reporte["auth"])
    return reporte

# 5. EJECUCIÓN (Aquí llamamos a todo)
# Asegúrate de que 'logs.txt' esté en la misma carpeta
# reporte_final = procesar_logs('logs.txt')
# print(json.dumps(reporte_final, indent=4))

In [24]:
import re
import json
from collections import defaultdict
from typing import List, Dict, Optional

# --- 1. PATRONES REGEX (Documentados con VERBOSE) ---
# Usamos lookbehind (?<=...) en AUTH como pide el reto.
PATRONES = {
    "HTTP": re.compile(r'''
        ^(?P<ip>\d{1,3}(?:\.\d{1,3}){3})\s+-\s+-\s+
        \[(?P<timestamp>[^\]]+)\]\s+
        "(?P<method>[A-Z]+)\s+(?P<path>[^\s]+)\s+HTTP/[0-9.]+"
        \s+(?P<status>\d{3})\s+(?P<bytes>\d+|-)
    ''', re.VERBOSE),
    
    "AUTH": re.compile(r'''
        ^\[AUTH\]\s+(?P<timestamp>[\d-]+\s[\d:]+)\s+\|\s+
        (?<=user=)(?P<user>[^\s|]+)\s+\|\s+
        (?<=action=)(?P<action>[^\s|]+)\s+\|\s+
        (?<=status=)(?P<status>[^\s|]+)
    ''', re.VERBOSE),
    
    "SQL_INJ": re.compile(r"OR\s+['\"]?\d+['\"]?\s*=\s*['\"]?\d+|UNION\s+SELECT|--|DROP\s+TABLE", re.IGNORECASE)
}

# --- 2. FUNCIONES DE PARSEO ---
def parsear_linea(linea: str) -> Dict:
    # Intenta clasificar la línea y extraer datos
    if "HTTP" in linea:
        m = PATRONES["HTTP"].search(linea)
        if m: 
            d = m.groupdict()
            d['status'] = int(d['status'])
            return {"tipo": "HTTP", "datos": d}
    elif "[AUTH]" in linea:
        m = PATRONES["AUTH"].search(linea)
        if m: return {"tipo": "AUTH", "datos": m.groupdict()}
    return None

# --- 3. ANALIZADOR DE SEGURIDAD ---
def ejecutar_analisis(reporte: Dict) -> Dict:
    # Fuerza bruta: IPs con más de 3 'FAILED'
    intentos = defaultdict(int)
    for log in [l['datos'] for l in reporte['auth_logs']]:
        if log['status'] == 'FAILED':
            intentos[log['datos']['ip']] += 1 # Ajustar según estructura
    
    # SQL Injection simple sobre un supuesto log de DB
    # (puedes expandir esto según los logs que cargues)
    return {"fuerza_bruta": [ip for ip, c in intentos.items() if c > 3]}

# --- 4. PROCESAMIENTO FINAL ---
def procesar_todo(archivo: str):
    reporte = {"http_logs": [], "auth_logs": [], "seguridad": {}}
    
    with open(archivo, 'r') as f:
        for linea in f:
            res = parsear_linea(linea.strip())
            if res:
                if res["tipo"] == "HTTP": reporte["http_logs"].append(res["datos"])
                if res["tipo"] == "AUTH": reporte["auth_logs"].append(res["datos"])
    
    reporte["seguridad"] = ejecutar_analisis(reporte)
    return reporte

# --- 5. EJECUCIÓN (Carga tu archivo aquí) ---
# resultado = procesar_todo('logs.txt')
# print(json.dumps(resultado, indent=4))

In [25]:
import re
import json
from collections import defaultdict

# 1. CREAMOS EL ARCHIVO DE PRUEBA (Para que tengas algo que procesar)
contenido = """[192.168.1.1] - - [18/Jun/2026:10:00:00] "GET /index.html HTTP/1.1" 200 1024
[AUTH] 2026-06-18 10:05:00 | user=admin | action=login | status=FAILED | ip=10.0.0.1
[AUTH] 2026-06-18 10:05:01 | user=admin | action=login | status=FAILED | ip=10.0.0.1
[AUTH] 2026-06-18 10:05:02 | user=admin | action=login | status=FAILED | ip=10.0.0.1
[AUTH] 2026-06-18 10:05:03 | user=admin | action=login | status=FAILED | ip=10.0.0.1
"""
with open('logs.txt', 'w') as f: f.write(contenido)

# 2. DEFINIMOS LOS PARSERS (La lógica)
def procesar_logs(nombre_archivo):
    reporte = {"http": [], "auth": [], "seguridad": {"fuerza_bruta": []}}
    
    patron_http = re.compile(r'(?P<ip>\d+\.\d+\.\d+\.\d+).*?"(?P<metodo>\w+) (?P<ruta>\S+)')
    patron_auth = re.compile(r'\[AUTH\].*?user=(?P<user>\S+).*?status=(?P<status>\S+).*?ip=(?P<ip>\S+)')
    
    intentos_fallidos = defaultdict(int)
    
    with open(nombre_archivo, 'r') as f:
        for linea in f:
            # Procesar HTTP
            m_http = patron_http.search(linea)
            if m_http: reporte["http"].append(m_http.groupdict())
            
            # Procesar AUTH
            m_auth = patron_auth.search(linea)
            if m_auth:
                data = m_auth.groupdict()
                reporte["auth"].append(data)
                if data['status'] == 'FAILED':
                    intentos_fallidos[data['ip']] += 1
    
    # 3. ANALIZAR SEGURIDAD
    reporte["seguridad"]["fuerza_bruta"] = [ip for ip, c in intentos_fallidos.items() if c > 3]
    return reporte

# 4. AHORA SÍ: AQUÍ ES DONDE IMPRIMIMOS EL RESULTADO
datos = procesar_logs('logs.txt')
print(json.dumps(datos, indent=4))

{
    "http": [
        {
            "ip": "192.168.1.1",
            "metodo": "GET",
            "ruta": "/index.html"
        }
    ],
    "auth": [
        {
            "user": "admin",
            "status": "FAILED",
            "ip": "10.0.0.1"
        },
        {
            "user": "admin",
            "status": "FAILED",
            "ip": "10.0.0.1"
        },
        {
            "user": "admin",
            "status": "FAILED",
            "ip": "10.0.0.1"
        },
        {
            "user": "admin",
            "status": "FAILED",
            "ip": "10.0.0.1"
        }
    ],
    "seguridad": {
        "fuerza_bruta": [
            "10.0.0.1"
        ]
    }
}
